## Zero Shot classification

In [2]:
# ! pip install git+https://github.com/huggingface/transformers@v4.56.0-Embedding-Gemma-preview
# ! pip install sentence-transformers
! pip install ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 12.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 13.1 MB/s  0:00:00

   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------

In [2]:
import torch
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, util
load_dotenv(r"../../.env")  # Load environment variables from .env file
hf_token = os.getenv("hugging_face_token")

In [3]:


# ── Complaints & labels ────────────────────────────────────────────────────────
complaints = [
    "I was charged twice for my purchase and customer care is not responding",
    "The delivery was delayed by 5 days and the package was damaged",
    "The product quality is very poor and not worth the price",
    "I want to return the item but the app is not allowing me to",
    "Payment failed but money got deducted from my account"
]

labels = [
    "billing issue",
    "delivery issue",
    "product quality",
    "return/refund",
    "payment issue",
    "customer service"
]

# ── Model 1: BART MNLI ─────────────────────────────────────────────────────────
print("Loading BART MNLI...")
bart = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", token = hf_token)

# ── Model 2: DistilBERT MNLI ───────────────────────────────────────────────────
print("Loading DistilBERT MNLI...")
distilbert = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli", token = hf_token)

# ── Model 3: Qwen 0.5B Base (likelihood scoring) ──────────────────────────────
print("Loading Qwen2.5-0.5B...")
model_id = "Qwen/Qwen2.5-0.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_id, token = hf_token)
qwen_model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32, token = hf_token).to(device)
qwen_model.eval()

def qwen_predict(complaint):
    scores = {}
    for label in labels:
        prompt = f"Complaint: {complaint}\nCategory: {label}"
        inputs = tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            outputs = qwen_model(**inputs, labels=inputs["input_ids"])
        scores[label] = -outputs.loss.item()
    best_label = max(scores, key=scores.get)
    # softmax for a comparable confidence score
    score_vals = torch.tensor(list(scores.values()))
    confidences = torch.softmax(score_vals, dim=0)
    best_conf = confidences[list(scores.keys()).index(best_label)].item()
    return best_label, best_conf

# ── Run all three & collect results ───────────────────────────────────────────
print("\nRunning classification...\n")
rows = []

for complaint in complaints:
    bart_result      = bart(complaint, candidate_labels=labels)
    distilbert_result = distilbert(complaint, candidate_labels=labels)
    qwen_label, qwen_conf = qwen_predict(complaint)

    rows.append({
        "Complaint"                  : complaint,
        "BART label"                 : bart_result["labels"][0],
        "BART conf"                  : round(bart_result["scores"][0], 4),
        "DistilBERT label"           : distilbert_result["labels"][0],
        "DistilBERT conf"            : round(distilbert_result["scores"][0], 4),
        "Qwen (decoder) label"       : qwen_label,
        "Qwen (decoder) conf"        : round(qwen_conf, 4),
    })

# ── Display ────────────────────────────────────────────────────────────────────
df = pd.DataFrame(rows)
print(df.to_string(index=False))
# df.to_excel("zero_shot_complaint_classification.xlsx", index=False)

Loading BART MNLI...


Device set to use cpu


Loading DistilBERT MNLI...


Device set to use cpu


Loading Qwen2.5-0.5B...

Running classification...

                                                              Complaint      BART label  BART conf DistilBERT label  DistilBERT conf Qwen (decoder) label  Qwen (decoder) conf
I was charged twice for my purchase and customer care is not responding   payment issue     0.4006 customer service           0.2591     customer service               0.2161
         The delivery was delayed by 5 days and the package was damaged  delivery issue     0.9406   delivery issue           0.9583     customer service               0.1989
               The product quality is very poor and not worth the price product quality     0.9279  product quality           0.9947      product quality               0.2248
            I want to return the item but the app is not allowing me to   return/refund     0.5672    payment issue           0.2862     customer service               0.2040
                  Payment failed but money got deducted from my account  

## Few Shot embedding with Sentence Transformer

Few shot embedding is nothing but giving some example to the prompt asking it to think in the same way.
Here we have used to two technique 
1. Using the example xml tag
2. using the sentence transformer we find the atleast 1 relevant example. If the relevant example is not there then it will take the closest match as possible

In [4]:
few_shot_examples = [
    ("I was charged extra on my bill", "billing issue"),
    ("My order arrived late and box was broken", "delivery issue"),
    ("The item stopped working in 2 days", "product quality"),
    ("Refund is not processed yet", "return/refund"),
    ("Money deducted but payment failed", "payment issue"),
]

In [5]:
embedder = SentenceTransformer('all-MiniLM-L6-v2', token = hf_token)
example_texts = [f"Complaint: {ex[0]} Category: {ex[1]}" for ex in few_shot_examples]
example_embeddings = embedder.encode(example_texts, convert_to_tensor=True)

In [6]:
query_embedding = embedder.encode("I want to return the item but the app is not allowing me", convert_to_tensor=True)
cosine_scores = util.cos_sim(query_embedding, example_embeddings)[0]
cosine_scores_numpy = cosine_scores.cpu().numpy()  # Move to CPU and convert to numpy
for i, (ex, score) in enumerate(zip(few_shot_examples, cosine_scores_numpy)):
    print(f"Example {i+1}: '{ex[0]}' (Category: {ex[1]}) - Cosine Similarity: {score:.4f}")

Example 1: 'I was charged extra on my bill' (Category: billing issue) - Cosine Similarity: 0.0982
Example 2: 'My order arrived late and box was broken' (Category: delivery issue) - Cosine Similarity: 0.2460
Example 3: 'The item stopped working in 2 days' (Category: product quality) - Cosine Similarity: 0.3452
Example 4: 'Refund is not processed yet' (Category: return/refund) - Cosine Similarity: 0.3921
Example 5: 'Money deducted but payment failed' (Category: payment issue) - Cosine Similarity: 0.1735


In [7]:
def few_shot_examples_identifier(query, top_k=3, threshold = 0.5):
    query_embedding = embedder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    cosine_scores = util.cos_sim(query_embedding, example_embeddings)[0]
    cosine_scores_numpy = cosine_scores.cpu().numpy()  # Move to CPU and convert to numpy
    # scored_examples = [(ex, score) for ex, score in zip(few_shot_examples, cosine_scores_numpy) if score >= threshold]
    scored_examples = []
    for i, (ex, score) in enumerate(zip(few_shot_examples, cosine_scores_numpy)):
        print(f"Example {i+1}: '{ex[0]}' (Category: {ex[1]}) - Cosine Similarity: {score:.4f}")
        if score >= threshold:
            scored_examples.append((ex, score))
    scored_examples.sort(key=lambda x: x[1], reverse=True)  # Sort by similarity score
    #fallback option if no examples meet the threshold
    if len(scored_examples) == 0:
        arr_position = cosine_scores_numpy.argmax()
        scored_examples.append((few_shot_examples[arr_position], cosine_scores_numpy[arr_position]))
    return scored_examples[:top_k][0][0]

In [8]:
a = few_shot_examples_identifier("I want to return the item but the app is not allowing me")

Example 1: 'I was charged extra on my bill' (Category: billing issue) - Cosine Similarity: 0.0982
Example 2: 'My order arrived late and box was broken' (Category: delivery issue) - Cosine Similarity: 0.2460
Example 3: 'The item stopped working in 2 days' (Category: product quality) - Cosine Similarity: 0.3452
Example 4: 'Refund is not processed yet' (Category: return/refund) - Cosine Similarity: 0.3921
Example 5: 'Money deducted but payment failed' (Category: payment issue) - Cosine Similarity: 0.1735


In [ ]:
few_shot_examples = [
    ("I was charged extra on my bill", "billing issue"),
    ("My order arrived late and box was broken", "delivery issue"),
    ("The item stopped working in 2 days", "product quality"),
    ("Refund is not processed yet", "return/refund"),
    ("Money deducted but payment failed", "payment issue"),
]
def few_shot_examples_identifier(query, top_k=3, threshold = 0.5):
    query_embedding = embedder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    cosine_scores = util.cos_sim(query_embedding, example_embeddings)[0]
    cosine_scores_numpy = cosine_scores.cpu().numpy()  # Move to CPU and convert to numpy
    # scored_examples = [(ex, score) for ex, score in zip(few_shot_examples, cosine_scores_numpy) if score >= threshold]
    scored_examples = []
    for i, (ex, score) in enumerate(zip(few_shot_examples, cosine_scores_numpy)):
        # print(f"Example {i+1}: '{ex[0]}' (Category: {ex[1]}) - Cosine Similarity: {score:.4f}")
        if score >= threshold:
            scored_examples.append((ex, score))
    scored_examples.sort(key=lambda x: x[1], reverse=True)  # Sort by similarity score
    #fallback option if no examples meet the threshold
    if len(scored_examples) == 0:
        arr_position = cosine_scores_numpy.argmax()
        scored_examples.append((few_shot_examples[arr_position], cosine_scores_numpy[arr_position]))
    return scored_examples[:top_k]

# ── Complaints & labels ────────────────────────────────────────────────────────
complaints = [
    "I was charged twice for my purchase and customer care is not responding",
    "The delivery was delayed by 5 days and the package was damaged",
    "The product quality is very poor and not worth the price",
    "I want to return the item but the app is not allowing me to",
    "Payment failed but money got deducted from my account"
]

labels = [
    "billing issue",
    "delivery issue",
    "product quality",
    "return/refund",
    "payment issue",
    "customer service"
]

# ── Model 1: BART MNLI ─────────────────────────────────────────────────────────
print("Loading BART MNLI...")
bart = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", token = hf_token)

# ── Model 2: DistilBERT MNLI ───────────────────────────────────────────────────
print("Loading DistilBERT MNLI...")
distilbert = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli", token = hf_token)

# ── Model 3: Qwen 0.5B Base (likelihood scoring) ──────────────────────────────
print("Loading Qwen2.5-0.5B...")
model_id = "Qwen/Qwen2.5-0.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_id, token = hf_token)
qwen_model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32, token = hf_token).to(device)
qwen_model.eval()

def qwen_predict(complaint):
    scores = {}
    examples = few_shot_examples_identifier(complaint)
    ex_prompt = ""
    for example in examples:
        ex_prompt += f"""
        <example_input>{example[0][0]}</example_input>
        <example_output>{example[0][1]}</example_output>"""
    for label in labels:
        prompt = f"""
        <Task>To do the text classification, assign the most appropriate category to the given complaint based on its content. Analyze the complaint carefully and determine which category it best fits into.</Task>
        <label>{label}</label>
        <input>{complaint}</input>
        {ex_prompt}
"""
        inputs = tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            outputs = qwen_model(**inputs, labels=inputs["input_ids"])
        scores[label] = -outputs.loss.item()
    best_label = max(scores, key=scores.get)
    # softmax for a comparable confidence score
    score_vals = torch.tensor(list(scores.values()))
    confidences = torch.softmax(score_vals, dim=0)
    best_conf = confidences[list(scores.keys()).index(best_label)].item()
    return best_label, best_conf

# ── Run all three & collect results ───────────────────────────────────────────
print("\nRunning classification...\n")
rows = []

for complaint in complaints:
    bart_result      = bart(complaint, candidate_labels=labels)
    distilbert_result = distilbert(complaint, candidate_labels=labels)
    qwen_label, qwen_conf = qwen_predict(complaint)

    rows.append({
        "Complaint"                  : complaint,
        "BART label"                 : bart_result["labels"][0],
        "BART conf"                  : round(bart_result["scores"][0], 4),
        "DistilBERT label"           : distilbert_result["labels"][0],
        "DistilBERT conf"            : round(distilbert_result["scores"][0], 4),
        "Qwen (decoder) label"       : qwen_label,
        "Qwen (decoder) conf"        : round(qwen_conf, 4),
    })

# ── Display ────────────────────────────────────────────────────────────────────
df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_excel("few_shot_complaint_classification.xlsx", index=False)

Loading BART MNLI...


Device set to use cpu


Loading DistilBERT MNLI...


Device set to use cpu


Loading Qwen2.5-0.5B...

Running classification...

                                                              Complaint      BART label  BART conf DistilBERT label  DistilBERT conf Qwen (decoder) label  Qwen (decoder) conf
I was charged twice for my purchase and customer care is not responding   payment issue     0.4006 customer service           0.2591        billing issue               0.1795
         The delivery was delayed by 5 days and the package was damaged  delivery issue     0.9406   delivery issue           0.9583       delivery issue               0.1878
               The product quality is very poor and not worth the price product quality     0.9279  product quality           0.9947      product quality               0.1846
            I want to return the item but the app is not allowing me to   return/refund     0.5672    payment issue           0.2862        return/refund               0.1852
                  Payment failed but money got deducted from my account  